# 10 - Research Report Agent

Demonstrates the **orchestrator + worker tools** pattern using `ResearchReportAgent`.

An orchestrator `SimpleAgent` autonomously coordinates three worker sub-agents exposed as MCP tools. The LLM decides which workers to call, in what order, and how many times — rather than following a predetermined code path.

```
research-report-agent  (orchestrator)
  ├─ [tool] research_overview   →  overview-worker     (SimpleAgent)
  ├─ [tool] find_examples       →  examples-worker     (SimpleAgent)
  └─ [tool] find_counterpoints  →  counterpoints-worker (SimpleAgent)
```

The agent class sets up all workers and their MCP server internally. No manual FastMCP wiring needed.

## Setup

Create an `OllamaClient` and pass it to `ResearchReportAgent`. The agent internally creates fresh client instances for each worker.

In [ ]:
from aimu.models.ollama import OllamaClient
from aimu.agents.examples import ResearchReportAgent

model_client = OllamaClient(OllamaClient.MODELS.QWEN_3_5_9B)
agent = ResearchReportAgent(model_client)

print("Orchestrator model:", model_client.model.value)
print("Worker tools:", [t.name for t in model_client.mcp_client.list_tools()])

## Run

Call `agent.run()` with a research topic. The orchestrator will call each worker tool, then synthesize the results into a final report.

In [ ]:
task = "What is retrieval-augmented generation?"

result = agent.run(task)
print(result)

Inspect the orchestrator's message history to see the full tool-calling sequence.

In [ ]:
agent.messages

## Streaming

`run(stream=True)` yields `AgentChunk` objects. `TOOL_CALLING` chunks show which worker was invoked and the worker's response; `GENERATING` chunks are the orchestrator's synthesis.

In [ ]:
from aimu.models import StreamPhase

task = "What is retrieval-augmented generation?"

for chunk in agent.run(task, stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        worker = chunk.content["name"]
        response_preview = chunk.content["response"][:80].replace("\n", " ")
        print(f"\n[tool: {worker}]\n  {response_preview}...\n")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## Clean Up

In [ ]:
del agent, model_client